# Práctico básico de Titanic — módulo IV

Primero cargué y revisé los datos de los pasajeros. Después completé las edades vacías, preparé unas pocas columnas y comparé dos modelos sencillos para estimar si sobrevivió cada persona.

In [ ]:
# Importé las librerías que usé para revisar, graficar y modelar.
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

## Cargué y revisé el CSV

Usé una copia pública del archivo de entrenamiento de Titanic.

In [ ]:
# Guardé el enlace del CSV de Titanic.
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

# Cargué los datos en una tabla de pandas.
df = pd.read_csv(url)

# Revisé el tamaño, las columnas, algunas filas y los vacíos.
print("Filas y columnas:", df.shape)
df.info()
display(df.head())
print(df.isnull().sum())

## Resumí las columnas numéricas con una función

La función guarda los pasos para poder usarlos de nuevo. La escribí con `def` y adentro usé `describe()` de pandas.

In [ ]:
# Definí una función para resumir las columnas numéricas.
def describir_numericas(datos):
    return datos.describe()

# Ejecuté la función para mirar ese resumen.
display(describir_numericas(df))

## Miré cuántas personas sobrevivieron

Primero conté las respuestas. Después comparé los resultados según el sexo y dibujé un gráfico sencillo.

In [ ]:
# Conté cuántas personas no sobrevivieron y cuántas sí.
print(df["Survived"].value_counts().sort_index())

# Comparé las respuestas por sexo.
print(df.groupby(["Sex", "Survived"])["PassengerId"].count().unstack(fill_value=0))

# Dibujé la cantidad de personas por resultado.
df["Survived"].value_counts().sort_index().plot(kind="bar")
plt.xticks([0, 1], ["No sobrevivió", "Sobrevivió"], rotation=0)
plt.ylabel("Cantidad de pasajeros")
plt.title("Pasajeros del Titanic según el resultado")
plt.show()

## Preparé algunas columnas

Usé cuatro datos para hacer una primera prueba: sexo, edad, clase del pasaje y si la persona viajaba sola. Completé las edades vacías con la mediana, que es el valor del medio cuando ordeno las edades.

In [ ]:
# Hice una copia para no cambiar el CSV original.
datos = df.copy()

# Completé las edades faltantes con la mediana.
datos["Age"] = datos["Age"].fillna(datos["Age"].median())

# Pasé el sexo a números para que los modelos lo puedan usar.
datos["Sex"] = datos["Sex"].map({"female": 1, "male": 0}).astype(int)

# Marqué con 1 a quienes viajaban sin hermanos, pareja, hijos o padres.
datos["FlagSolo"] = ((datos["SibSp"] == 0) & (datos["Parch"] == 0)).astype(int)

print(datos[["Sex", "Age", "Pclass", "FlagSolo", "Survived"]].head())

## Separé los datos y entrené los modelos

Separé una parte para entrenar y otra para revisar las predicciones. Probé Regresión Logística y Árbol de Decisión, que son los dos modelos que aparecen en el módulo.

In [ ]:
# Elegí las columnas de entrada y la respuesta que quería estimar.
X = datos[["Sex", "Age", "Pclass", "FlagSolo"]]
y = datos["Survived"]

# Separé los datos para no revisar el modelo con las mismas filas usadas para entrenarlo.
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Creé y entrené los dos modelos.
modelos = {
    "Regresión Logística": LogisticRegression(max_iter=1000, random_state=42),
    "Árbol de Decisión": DecisionTreeClassifier(random_state=42),
}

for modelo in modelos.values():
    modelo.fit(X_entrenamiento, y_entrenamiento)

## Revisé los resultados y guardé unas predicciones

La matriz de confusión muestra cuántas predicciones acertó o erró cada modelo. Guardé en un CSV las respuestas reales y las predichas para poder mirarlas.

In [ ]:
# Calculé la exactitud y la matriz de confusión de cada modelo.
predicciones = {}

for nombre, modelo in modelos.items():
    prediccion = modelo.predict(X_prueba)
    predicciones[nombre] = prediccion
    print("\n", nombre)
    print("Accuracy:", round(accuracy_score(y_prueba, prediccion), 3))
    print("Matriz de confusión:")
    print(confusion_matrix(y_prueba, prediccion))

# Guardé las predicciones de la parte que separé para la prueba.
salida = pd.DataFrame({
    "PassengerId": datos.loc[X_prueba.index, "PassengerId"],
    "sobrevivio_real": y_prueba,
    "prediccion_logistica": predicciones["Regresión Logística"],
    "prediccion_arbol": predicciones["Árbol de Decisión"],
})
salida.to_csv("predicciones_titanic.csv", index=False)
print("\nGuardé predicciones_titanic.csv")
display(salida.head())

## Qué revisé

El archivo de salida deja comparar la respuesta real con las dos predicciones. La exactitud y la matriz de confusión me ayudan a ver cómo funcionaron estos modelos en las filas que separé. Es un ejercicio del módulo IV; no es el proyecto de concentración delictiva de PP1.